In [1]:
import os
# --- FIX 1: PREVENT WINDOWS/INTEL DRIVER CRASHES ---
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
import torch.nn.functional as F
import cv2
import numpy as np
import matplotlib.pyplot as plt
from torchvision import models, transforms
from PIL import Image

# ==========================================
# 1. CLASS DEFINITION
# ==========================================
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self.target_layer.register_forward_hook(self.save_activation)
        self.target_layer.register_full_backward_hook(self.save_gradient)

    def save_activation(self, module, input, output):
        self.activations = output

    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def __call__(self, x, class_idx=None):
        output = self.model(x)
        if class_idx is None:
            class_idx = torch.argmax(output, dim=1).item()
        self.model.zero_grad()
        one_hot = torch.zeros_like(output)
        one_hot[0][class_idx] = 1
        output.backward(gradient=one_hot, retain_graph=True)
        
        pooled_gradients = torch.mean(self.gradients, dim=[0, 2, 3])
        activations = self.activations.detach().clone()
        for i in range(activations.shape[1]):
            activations[:, i, :, :] *= pooled_gradients[i]
            
        heatmap = torch.mean(activations, dim=1).squeeze()
        heatmap = F.relu(heatmap)
        if torch.max(heatmap) > 0:
            heatmap /= torch.max(heatmap)
        return heatmap.cpu().numpy()

def overlay_heatmap(heatmap, original_image, alpha=0.4, colormap=cv2.COLORMAP_JET):
    heatmap = cv2.resize(heatmap, (original_image.size[0], original_image.size[1]))
    heatmap_uint8 = np.uint8(255 * heatmap)
    heatmap_colored = cv2.applyColorMap(heatmap_uint8, colormap)
    heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)
    original_np = np.array(original_image)
    return cv2.addWeighted(original_np, 1 - alpha, heatmap_colored, alpha, 0)

# ==========================================
# 2. MAIN EXECUTION
# ==========================================

# --- CONFIGURATION ---
IMAGE_PATHS = [
    'C:/Users/deyko/Desktop/dbrd/test_images_512/test_images_512/34_left.jpg',
    'C:/Users/deyko/Desktop/dbrd/test_images_512/test_images_512/34_right.jpg'
]
MODEL_ARCH = 'efficientnet' 
device = torch.device("cpu") # Explicitly set to CPU to match your log

# Load Model
print("Loading Model...")
if MODEL_ARCH == 'resnet':
    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    target_layer = model.layer4[-1]
elif MODEL_ARCH == 'efficientnet':
    model = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.DEFAULT)
    target_layer = model.features[-1]

model.to(device)
model.eval()

# Preprocess
preprocess = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

grad_cam = GradCAM(model, target_layer)
results = []

print("Processing Images...")

for img_path in IMAGE_PATHS:
    try:
        raw_img = Image.open(img_path).convert('RGB')
        input_tensor = preprocess(raw_img).unsqueeze(0).to(device)
        heatmap = grad_cam(input_tensor)
        overlay = overlay_heatmap(heatmap, raw_img)
        
        with torch.no_grad():
            output = model(input_tensor)
            pred_idx = torch.argmax(output, dim=1).item()
            
        results.append({"original": raw_img, "overlay": overlay, "path": img_path, "pred": pred_idx})
        print(f"✅ Processed {os.path.basename(img_path)}")
        
    except Exception as e:
        print(f"❌ Error: {e}")

# ==========================================
# 3. SAVE RESULT INSTEAD OF SHOWING (Prevents Crash)
# ==========================================
if len(results) >= 2:
    print("Creating output grid...")
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    
    # Image 1
    axes[0, 0].imshow(results[0]["original"])
    axes[0, 0].set_title(f"Left Eye Original (Pred: {results[0]['pred']})")
    axes[0, 0].axis('off')
    axes[0, 1].imshow(results[0]["overlay"])
    axes[0, 1].set_title("Left Eye Grad-CAM")
    axes[0, 1].axis('off')

    # Image 2
    axes[1, 0].imshow(results[1]["original"])
    axes[1, 0].set_title(f"Right Eye Original (Pred: {results[1]['pred']})")
    axes[1, 0].axis('off')
    axes[1, 1].imshow(results[1]["overlay"])
    axes[1, 1].set_title("Right Eye Grad-CAM")
    axes[1, 1].axis('off')

    plt.tight_layout()
    
    # --- SAVE FILE ---
    output_path = 'gradcam_result.png'
    plt.savefig(output_path)
    print(f"\n✅ SUCCESS! Image saved to: {os.path.abspath(output_path)}")
    print("Go open this file to see your results.")
    
    # Close memory to prevent leaks
    plt.close(fig)
else:
    print("Not enough results to plot.")

Loading Model...
Processing Images...
✅ Processed 34_left.jpg
✅ Processed 34_right.jpg
Creating output grid...

✅ SUCCESS! Image saved to: c:\Users\deyko\Desktop\newdbrdidea\gradcam_result.png
Go open this file to see your results.
